# Step 9: Neighbourhood features (idea G)
**Idea:** buildings of the same era cluster in neighbourhoods. Smoothing already uses this AFTER prediction. Here the model gets it as INPUT: next to its own 132 features, every pixel also gets the **average features of the pixels around it** (3x3 window = 90 m, and in variant G2 also a 7x7 window = 210 m), plus how many neighbours it has. The organisers' Notebook 1 explicitly invites "spatial context".

**No leakage:** neighbour averages use satellite FEATURES only, never labels. At test time the neighbours are the test pixels plus our own pixels of that city (the same context files smoothing already uses), so it works whether the test pixels are a separate area or scattered among ours.

**Same rules as notebook 08 (fixed in advance):** screening on the 80% training split only, paired seeds; pass if n=25 or the 5-n average improves by at least +0.005 and no n gets worse by more than 0.005, and Madrid random CV stays within -0.005; the winner is confirmed ONCE on the hidden neighbourhoods.

`BASE_CFG` below is the starting setup: leave `{}` (the frozen method) unless notebook 08 adopted something; then put 08's adopted setup there so G is tested on top of it.

Writes only to `results/exp09/`. Run time: about 60 to 90 minutes.

In [1]:
import time, importlib, json
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score
import data_prep as dp, features_v3 as fv3, transfer_v3 as tv3, experiments_v4 as ex
for m in (dp, fv3, tv3, ex):
    importlib.reload(m)

BASE_CFG = {}                       # e.g. {"align": True} if notebook 08 adopted idea D
BASE_PRIOR, BASE_SMOOTH = False, (0.5, 2)   # change only if 08 adopted C / another smoothing
VARIANTS = {"G1_3x3": (1,), "G2_3x3_7x7": (1, 3)}
SHOTS, TRIALS = (5, 25, 50, 100, 200), 10
SPATIAL_BLOCK = 50
SEED, CLASSES = tv3.SEED, tv3.CLASSES
R = Path("rehearsal")
for DATA_DIR in [Path("data"), Path("../data"), Path("../../data")]:
    if (DATA_DIR / "madrid_train.parquet").exists():
        break
OUT = Path("results/exp09"); OUT.mkdir(parents=True, exist_ok=True)

D0 = ex.load(R, DATA_DIR)                      # 80% split, own features only

def with_neigh(D, radii):
    """Copy of D where Xm / Xa also carry neighbourhood features (each city uses its own pixels as neighbours)."""
    G = dict(D)
    G["Xm"] = ex.add_neighbourhood(D["Xm"], D["ids_m"], radii)
    G["Xa"] = ex.add_neighbourhood(D["Xa"], D["ids_a"], radii)
    return G

DATASETS = {"baseline": D0, **{name: with_neigh(D0, radii) for name, radii in VARIANTS.items()}}
for name, Dx in DATASETS.items():
    print(f"{name:12s} features per pixel: {Dx['Xm'].shape[1]}")

baseline     features per pixel: 132
G1_3x3       features per pixel: 265
G2_3x3_7x7   features per pixel: 398


## 1. Madrid: random and spatial CV

In [2]:
t0 = time.time()
rf_params = BASE_CFG.get("rf_params", tv3.RF_PARAMS)
mrows = []
for name, Dx in DATASETS.items():
    t = time.time()
    r = ex.madrid_cv(Dx, rf_params, weights=BASE_CFG.get("weights", False), block=SPATIAL_BLOCK)
    Ps = tv3.smooth_proba(r["P_spatial"], Dx["ids_m"], alpha=0.5, r=1)
    mrows.append({"config": name, "random_cv": r["random"][0], "spatial_cv": r["spatial"][0],
                  "spatial_smoothed": f1_score(Dx["ym"], CLASSES[Ps.argmax(1)], average="macro")})
    print(f"{name:12s} random {r['random'][0]:.4f} | spatial {r['spatial'][0]:.4f} | spatial+smooth {mrows[-1]['spatial_smoothed']:.4f}  ({time.time()-t:.0f}s)")
mad = pd.DataFrame(mrows).set_index("config")
mad.round(4).to_csv(OUT / "madrid_cv.csv")
print(f"done in {time.time()-t0:.0f}s")

baseline     random 0.6623 | spatial 0.6165 | spatial+smooth 0.6323  (125s)
G1_3x3       random 0.6951 | spatial 0.6243 | spatial+smooth 0.6275  (181s)
G2_3x3_7x7   random 0.7384 | spatial 0.6337 | spatial+smooth 0.6354  (220s)
done in 526s


## 2. Amsterdam transfer screening (paired seeds)

In [3]:
t0 = time.time()
rows = []
for name, Dx in DATASETS.items():
    t = time.time()
    S = ex.build_stage1(Dx, BASE_CFG)
    for n in SHOTS:
        for k in range(TRIALS):
            sup, qry, P = ex.transfer_run(Dx, S, n, SEED + k)
            f = ex.scores(P, Dx["ya"][qry], Dx["ids_a"][qry], prior=BASE_PRIOR)
            rows.append({"config": name, "n_per_class": n, "seed": SEED + k,
                         "f1_raw": f[(0.0, 1)], "f1": f[BASE_SMOOTH]})
    g = pd.DataFrame(rows).query("config == @name").groupby("n_per_class").f1.mean()
    print(f"{name:12s} " + " | ".join(f"n={n}: {g[n]:.4f}" for n in SHOTS) + f" | avg {g.mean():.4f}  ({time.time()-t:.0f}s)")
runs = pd.DataFrame(rows)
runs.round(5).to_csv(OUT / "transfer_screening_all_runs.csv", index=False)
print(f"done in {time.time()-t0:.0f}s")

baseline     n=5: 0.5890 | n=25: 0.6685 | n=50: 0.7011 | n=100: 0.7178 | n=200: 0.7336 | avg 0.6820  (447s)
G1_3x3       n=5: 0.6028 | n=25: 0.6747 | n=50: 0.7011 | n=100: 0.7168 | n=200: 0.7368 | avg 0.6864  (652s)
G2_3x3_7x7   n=5: 0.6037 | n=25: 0.6822 | n=50: 0.7075 | n=100: 0.7252 | n=200: 0.7434 | avg 0.6924  (794s)
done in 1894s


In [4]:
def compare(new, base_df, label):
    m = new.merge(base_df, on=["n_per_class", "seed"], suffixes=("", "_base"))
    m["diff"] = m.f1 - m.f1_base
    g = m.groupby("n_per_class").agg(new=("f1", "mean"), diff=("diff", "mean"), wins=("diff", lambda d: int((d > 0).sum())))
    ok = ((g.loc[25, "diff"] >= 0.005) or (g["diff"].mean() >= 0.005)) and (g["diff"].min() >= -0.005)
    return {"idea": label, **{f"diff_n{n}": g.loc[n, "diff"] for n in SHOTS}, "avg_diff": g["diff"].mean(),
            "wins_n25": g.loc[25, "wins"], "avg_new": g["new"].mean(), "rule_pass": bool(ok)}

pick = lambda c: runs[runs.config == c][["n_per_class", "seed", "f1"]]
verdict = pd.DataFrame([compare(pick(v), pick("baseline"), v) for v in VARIANTS])
for i, r in verdict.iterrows():                       # Madrid gate
    verdict.loc[i, "rule_pass"] &= bool(mad.loc[r.idea, "random_cv"] >= mad.loc["baseline", "random_cv"] - 0.005)
display(verdict.round(4))
verdict.round(4).to_csv(OUT / "verdict.csv", index=False)
passed = verdict[verdict.rule_pass].sort_values("avg_new")
G_CHOICE = passed.iloc[-1].idea if len(passed) else None
print("PASSED:", G_CHOICE or "none -> neighbourhood features not adopted")

,idea,diff_n5,diff_n25,diff_n50,diff_n100,diff_n200,avg_diff,wins_n25,avg_new,rule_pass
0,G1_3x3,0.0138,0.0062,0.0000,-0.0010,0.0032,0.0045,8,0.6864,True
1,G2_3x3_7x7,0.0147,0.0137,0.0065,0.0074,0.0099,0.0104,10,0.6924,True


PASSED: G2_3x3_7x7


## 3. Confirm ONCE on the hidden neighbourhoods
Hidden pixels get neighbourhood features from the hidden pixels PLUS the training pixels of the same city (exactly what predict.py would do with context files). Training pixels keep the features they were trained with. The baseline must reproduce notebook 05 (sanity check).

In [5]:
def hidden_own(city):
    raw = dp.load_city(R / f"hidden_{city.lower()}.parquet")
    ev = {"Madrid": 1960, "Amsterdam": 1945}[city]
    f = fv3.build_features(dp.add_age_class(dp.clean_long_table(raw, combine="median"), ev))[0]
    return f[D0["cols"]].to_numpy(float), f["age_class"].to_numpy(int), f[["px_key", "py_key"]].to_numpy(np.int64)

def hidden_eval(name):
    Dx = DATASETS[name]
    radii = VARIANTS.get(name)
    def aug(Xh, ids_h, Xtrain_own, ids_train):
        if radii is None:
            return Xh
        F = np.vstack([Xh, Xtrain_own]); I = np.vstack([ids_h, ids_train])
        return ex.add_neighbourhood(F, I, radii, n_own=len(Xh))
    XhA = aug(XhA_own, idsA, D0["Xa"], D0["ids_a"])
    XhM = aug(XhM_own, idsM, D0["Xm"], D0["ids_m"])
    S = ex.build_stage1(Dx, BASE_CFG)
    rows = []
    for n in SHOTS:
        for k in range(TRIALS):
            sup, qry, _, parts = ex.transfer_run(Dx, S, n, SEED + k, return_parts=True)
            Ph = ex.predict_new(S, parts, XhA)
            Pc = ex.predict_new(S, parts, Dx["Xa"])
            if BASE_PRIOR:
                Ph = ex.prior_correct(Ph)[0]
            P = tv3.smooth_proba(np.vstack([Ph, Pc]), np.vstack([idsA, Dx["ids_a"]]), alpha=BASE_SMOOTH[0], r=BASE_SMOOTH[1])[: len(Ph)]
            rows.append({"setup": name, "n_per_class": n, "seed": SEED + k,
                         "f1_raw": f1_score(yA, CLASSES[Ph.argmax(1)], average="macro"),
                         "f1": f1_score(yA, CLASSES[P.argmax(1)], average="macro")})
    Pm, Pmc = S["rf"].predict_proba(XhM), S["rf"].predict_proba(Dx["Xm"])
    Pms = tv3.smooth_proba(np.vstack([Pm, Pmc]), np.vstack([idsM, Dx["ids_m"]]), alpha=0.5, r=1)[: len(Pm)]
    return pd.DataFrame(rows), f1_score(yM, CLASSES[Pms.argmax(1)], average="macro")

if G_CHOICE:
    t0 = time.time()
    XhA_own, yA, idsA = hidden_own("Amsterdam")
    XhM_own, yM, idsM = hidden_own("Madrid")
    hb, mb = hidden_eval("baseline")
    if not BASE_CFG and not BASE_PRIOR:
        ref = pd.read_csv(R / "dress_rehearsal_all_runs.csv")[["n_per_class", "seed", "f1_hidden"]]
        gap = hb.merge(ref, on=["n_per_class", "seed"]).eval("f1_raw - f1_hidden").abs().max()
        print(f"sanity check vs notebook 05 hidden: largest difference {gap:.6f}")
        assert gap < 1e-3
    hg, mg = hidden_eval(G_CHOICE)
    conf = compare(hg[["n_per_class", "seed", "f1"]], hb[["n_per_class", "seed", "f1"]], G_CHOICE)
    display(pd.concat([hb, hg]).groupby(["setup", "n_per_class"]).f1.agg(["mean", "std"]).unstack(0).round(4))
    pd.concat([hb, hg]).round(5).to_csv(OUT / "hidden_confirmation_all_runs.csv", index=False)
    confirmed = conf["avg_diff"] >= 0 and conf["diff_n25"] >= -0.005 and conf["diff_n200"] >= -0.005
    print(f"Madrid hidden + smoothing: baseline {mb:.4f} | {G_CHOICE} {mg:.4f}")
    print(f"Amsterdam hidden average: baseline {hb.groupby('n_per_class').f1.mean().mean():.4f} | "
          f"{G_CHOICE} {hg.groupby('n_per_class').f1.mean().mean():.4f} | n=25 diff {conf['diff_n25']:+.4f} (wins {conf['wins_n25']}/{TRIALS})")
    print("FINAL VERDICT:", f"ADOPT {G_CHOICE}" if confirmed else "NOT CONFIRMED -> do not adopt")
    json.dump({"choice": G_CHOICE, "confirmed": bool(confirmed), "madrid_hidden_base": mb, "madrid_hidden_new": mg},
              open(OUT / "final_verdict.json", "w"), indent=2)
    print(f"done in {time.time()-t0:.0f}s")

temporal haze filter removed 930 photos (0.31% of clean photos)
built 3,477 pixels x 132 features (0 NaN values filled with the city median)
temporal haze filter removed 5,485 photos (0.33% of clean photos)
built 14,532 pixels x 132 features (0 NaN values filled with the city median)
sanity check vs notebook 05 hidden: largest difference 0.000542


mean                 std         
setup       G2_3x3_7x7 baseline G2_3x3_7x7 baseline
n_per_class                                        
5               0.5324   0.5177     0.0524   0.0445
25              0.6032   0.5907     0.0391   0.0331
50              0.6364   0.6277     0.0139   0.0175
100             0.6511   0.6462     0.0154   0.0098
200             0.6615   0.6521     0.0062   0.0064

Madrid hidden + smoothing: baseline 0.6088 | G2_3x3_7x7 0.6059
Amsterdam hidden average: baseline 0.6069 | G2_3x3_7x7 0.6169 | n=25 diff +0.0125 (wins 9/10)
FINAL VERDICT: ADOPT G2_3x3_7x7
done in 1329s
